# Prepare shared CausalRivers discharge measurements data

This notebook creates or verifies the frozen `causalRivers` bundles used by
all seven method notebooks in this directory. Data generation and acquisition
are kept separate from model fitting so every method reads byte-identical
observations and the same optional contextual reference network.

This case study uses measured river-discharge time series. Five connected five-station subgraphs are selected deterministically from each of the eastern-Germany and Bavaria networks using the authors' sampling routine. The constructed river network is visual context, not a required exhaustive causal scoring target.

In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import math
import os
import platform
import subprocess
import sys
import tempfile
import types
import urllib.request
import zipfile
import pickle
from pathlib import Path

EXPECTED_PYTHON = (3, 14)
if sys.version_info[:2] != EXPECTED_PYTHON:
    raise RuntimeError(
        "This notebook requires CPython 3.14; "
        f"found {platform.python_version()}."
    )

import numpy as np
import pandas as pd
import scipy.special

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "src" / "fastCGC" / "__init__.py").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the fastC-GC project root.")
SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from fastCGC.experiments.upstream import verify_upstream_revision

In [ ]:
def series_digest(series):
    return hashlib.sha256(
        np.ascontiguousarray(series, dtype=np.float64).tobytes()
    ).hexdigest()


def save_bundle(
    path,
    *,
    series,
    metadata,
    aggregate_truth=None,
    lag_truth=None,
    reference_graph=None,
    extra_arrays=None,
):
    graph_arrays = (aggregate_truth, lag_truth, reference_graph)
    if sum(value is not None for value in graph_arrays) > 1:
        raise ValueError("Provide at most one truth or reference graph.")
    path.parent.mkdir(parents=True, exist_ok=True)
    arrays = {"series": np.asarray(series, dtype=float)}
    if lag_truth is not None:
        arrays["lag_truth"] = np.asarray(lag_truth, dtype=bool)
    elif aggregate_truth is not None:
        arrays["aggregate_truth"] = np.asarray(
            aggregate_truth,
            dtype=bool,
        )
    elif reference_graph is not None:
        arrays["reference_graph"] = np.asarray(
            reference_graph,
            dtype=bool,
        )
    for name, value in (extra_arrays or {}).items():
        if name in arrays:
            raise ValueError(f"Duplicate bundle array name: {name}.")
        arrays[name] = np.asarray(value)
    digest = series_digest(arrays["series"])
    metadata = {**metadata, "data_digest": digest}
    metadata_path = path.with_suffix(".json")
    if path.exists() or metadata_path.exists():
        if not path.is_file() or not metadata_path.is_file():
            raise FileNotFoundError(f"Incomplete dataset bundle at {path}.")
        recorded = json.loads(metadata_path.read_text(encoding="utf-8"))
        with np.load(path, allow_pickle=False) as stored:
            saved = {name: np.asarray(stored[name]) for name in stored.files}
        arrays_match = set(saved) == set(arrays) and all(
            np.array_equal(saved[name], value)
            for name, value in arrays.items()
        )
        if not arrays_match or recorded != metadata:
            raise ValueError(f"Existing bundle does not match request: {path}.")
        return False

    with tempfile.NamedTemporaryFile(
        mode="wb", dir=path.parent, prefix=f".{path.name}.", delete=False
    ) as handle:
        np.savez_compressed(handle, **arrays)
        temporary_data = Path(handle.name)
    os.replace(temporary_data, path)

    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        dir=path.parent,
        prefix=f".{metadata_path.name}.",
        delete=False,
    ) as handle:
        json.dump(metadata, handle, indent=2, sort_keys=True)
        handle.write("\n")
        temporary_metadata = Path(handle.name)
    os.replace(temporary_metadata, metadata_path)
    return True


def file_digest(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_verified(url, path, expected_sha256):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.is_file():
        observed = file_digest(path)
        if observed != expected_sha256:
            raise ValueError(
                f"File digest mismatch for {path}: {observed}."
            )
        return False

    with tempfile.NamedTemporaryFile(
        mode="wb",
        dir=path.parent,
        prefix=f".{path.name}.",
        delete=False,
    ) as handle:
        temporary = Path(handle.name)
        with urllib.request.urlopen(url, timeout=120) as response:
            while chunk := response.read(1024 * 1024):
                handle.write(chunk)
    try:
        observed = file_digest(temporary)
        if observed != expected_sha256:
            raise ValueError(
                f"Downloaded file digest mismatch: {observed}."
            )
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()
    return True

In [ ]:
DATASET_KEY = "causalRivers"
DATASET_DIR = (
    PROJECT_ROOT / "data" / "shared" / "causal-comparison"
    / DATASET_KEY / "datasets"
)
UPSTREAM_ROOT = Path(
    os.environ.get(
        "CAUSAL_RIVERS_ROOT",
        PROJECT_ROOT / "external" / "causalrivers",
    )
).expanduser()
UPSTREAM_REVISION = verify_upstream_revision(
    UPSTREAM_ROOT,
    expected_url="https://github.com/CausalRivers/causalrivers.git",
    expected_commit="a30154fbb9a2e03bae133c643d7e978df63b3766",
)
if str(UPSTREAM_ROOT) not in sys.path:
    sys.path.insert(0, str(UPSTREAM_ROOT))
try:
    import networkx as nx
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        "CausalRivers preparation requires networkx from the official "
        "CausalRivers environment. Install that environment before running "
        "this notebook; networkx is not a fastC-GC estimator dependency."
    ) from error
from tools.graph_sampling_tools import get_all_subgraphs

ARCHIVE_PATH = Path(
    os.environ.get(
        "CAUSAL_RIVERS_ARCHIVE",
        PROJECT_ROOT / "external" / "datasets" / "causalrivers-product.zip",
    )
).expanduser()
download_verified(
    "https://github.com/CausalRivers/causalrivers/releases/download/First_release/product.zip",
    ARCHIVE_PATH,
    "4f779fa9222653af092046eacba39a955cc65638bfb70fb7e31a32d9d8373b55",
)

REGIONS = {
    "east-germany": (
        "product/rivers_east_germany.p",
        "product/rivers_ts_east_germany.csv",
    ),
    "bavaria": (
        "product/rivers_bavaria.p",
        "product/rivers_ts_bavaria.csv",
    ),
}
NODES_PER_SUBGRAPH = 5
SUBGRAPHS_PER_REGION = 5
FIT_ORDER = 5
RESOLUTION = "6h"
MAX_NEW_DATASETS: int | None = None


def deterministic_subgraphs(graph):
    candidates = sorted(
        tuple(sorted(int(node) for node in candidate))
        for candidate in get_all_subgraphs(graph, n_vars=NODES_PER_SUBGRAPH)
    )
    if len(candidates) < SUBGRAPHS_PER_REGION:
        raise ValueError(
            f"Only {len(candidates)} candidate subgraphs were found."
        )
    indices = np.linspace(
        0,
        len(candidates) - 1,
        SUBGRAPHS_PER_REGION,
        dtype=int,
    )
    return candidates, [(int(index), candidates[int(index)]) for index in indices]


created = 0
stop_requested = False
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    for region, (graph_member, series_member) in REGIONS.items():
        # The archive hash is checked before loading this official NetworkX pickle.
        graph = pickle.loads(archive.read(graph_member))
        candidates, selections = deterministic_subgraphs(graph)
        required_nodes = sorted({
            node for _, nodes in selections for node in nodes
        })
        with archive.open(series_member) as handle:
            raw = pd.read_csv(
                handle,
                index_col="datetime",
                usecols=["datetime", *map(str, required_nodes)],
            )
        raw["dt"] = pd.to_datetime(raw.index).round(RESOLUTION).values
        processed = raw.groupby("dt").mean()
        minima = processed.min(axis=0)
        spans = processed.max(axis=0) - minima
        if spans.isna().any() or (spans <= 0).any():
            invalid = spans.index[spans.isna() | (spans <= 0)].tolist()
            raise ValueError(f"Unusable river stations in {region}: {invalid}.")
        processed = (processed - minima) / spans
        processed = processed.interpolate()

        for selection_rank, (candidate_index, nodes) in enumerate(selections):
            if MAX_NEW_DATASETS is not None and created >= MAX_NEW_DATASETS:
                stop_requested = True
                break
            names = [str(node) for node in nodes]
            sample = processed[names]
            complete_rows = np.flatnonzero(~sample.isna().any(axis=1).to_numpy())
            if not len(complete_rows):
                raise ValueError(
                    f"No complete observations for {region} nodes {nodes}."
                )
            sample = sample.iloc[complete_rows[0] : complete_rows[-1] + 1]
            if sample.isna().any().any():
                raise ValueError(
                    f"Interior missing values remain for {region} nodes {nodes}."
                )

            node_index = {node: index for index, node in enumerate(nodes)}
            reference_graph = np.zeros(
                (NODES_PER_SUBGRAPH, NODES_PER_SUBGRAPH),
                dtype=bool,
            )
            for source, target in graph.subgraph(nodes).edges:
                reference_graph[node_index[source], node_index[target]] = True
            positions = np.asarray(
                [graph.nodes[node].get("p", (np.nan, np.nan)) for node in nodes],
                dtype=float,
            )
            dataset_id = (
                f"causal-rivers-{region}-n{NODES_PER_SUBGRAPH}-"
                f"sample{selection_rank + 1:02d}"
            )
            was_created = save_bundle(
                DATASET_DIR / f"{dataset_id}.npz",
                series=sample.to_numpy(dtype=float).T,
                reference_graph=reference_graph,
                extra_arrays={"node_positions": positions},
                metadata={
                    "dataset_id": dataset_id,
                    "dataset": DATASET_KEY,
                    "data_source": "https://github.com/CausalRivers/causalrivers/releases/download/First_release/product.zip",
                    "data_archive_sha256": "4f779fa9222653af092046eacba39a955cc65638bfb70fb7e31a32d9d8373b55",
                    "data_license": "Data licence Germany (dl-de)",
                    "sampling_source": "https://github.com/CausalRivers/causalrivers.git",
                    "upstream_commit": UPSTREAM_REVISION.commit,
                    "region": region,
                    "regime": region,
                    "fit_order": FIT_ORDER,
                    "resolution": RESOLUTION,
                    "variable_names": names,
                    "node_ids": list(nodes),
                    "candidate_index": candidate_index,
                    "candidate_count": len(candidates),
                    "selection_rank": selection_rank,
                    "sampling_strategy": (
                        "deterministic evenly spaced selection from the sorted "
                        "output of the authors' get_all_subgraphs function"
                    ),
                    "preprocessing": (
                        "6-hour rounded means, per-station min-max normalization, "
                        "linear interpolation, and removal of incomplete end rows"
                    ),
                    "n_variables": NODES_PER_SUBGRAPH,
                    "n_samples": int(sample.shape[0]),
                    "series_layout": "variable, time",
                    "reference_layout": "source, target",
                    "reference_scope": (
                        "constructed river-connectivity network supplied with "
                        "CausalRivers; used as a contextual graph rather than a "
                        "mandatory exhaustive causal scoring target"
                    ),
                    "evaluation": "qualitative inferred-graph comparison",
                },
            )
            created += int(was_created)
        if stop_requested:
            break
print(
    f"Created {created} new CausalRivers bundles. "
    "Rerun to verify existing bundles and resume."
)